# Chapter 5 &mdash; Four Ways to Specify a Language

**Concept 1 of the Chapter 5 decomposition:** *Four Ways to Specify a Language, and Why You Need More Than One*

English, set comprehension, positive/negative examples, and a different perspective &mdash; cross-check them before you draw anything.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter5/Concept-Four-Ways-To-Specify/Concept-Four-Ways-To-Specify.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.LangDef        import *
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


A DFA is only as good as the specification it was built from, and **English alone is
not enough**. Use four views and make them agree:

1. **English** &mdash; readable, but ambiguous;
2. **set comprehension** &mdash; precise, but easy to get subtly wrong;
3. **positive and negative examples** &mdash; especially the short and empty ones;
4. **a different perspective** &mdash; restate the same language another way.

Disagreement between views is a **bug found before you draw the machine**, which is
far cheaper than finding it after.

## 2. Definitions

### The language, four ways

$L$ = strings over $\{0,1\}$ with an even number of 0s **and** an even number of 1s.

In [ ]:
# View 1 -- English (above).
# View 2 -- set comprehension:
def in_L_setcomp(s):
    return s.count('0') % 2 == 0 and s.count('1') % 2 == 0

# View 4 -- a different perspective: even length AND even number of 0s
def in_L_other(s):
    return len(s) % 2 == 0 and s.count('0') % 2 == 0

### View 3 &mdash; examples, with the short ones written down first

In [ ]:
positives = ['', '00', '11', '0011', '0101', '1100', '1001']
negatives = ['0', '1', '01', '10', '000', '111', '011']

### An enumerator, so all four can be cross-checked mechanically

In [ ]:
from itertools import product
def upto(n, sigma='01'):
    return [''.join(p) for k in range(n+1) for p in product(sigma, repeat=k)]

## 3. Tests

Views 2 and 4 agree &mdash; the different perspective corroborates the comprehension.

In [ ]:
bad = [s for s in upto(10) if in_L_setcomp(s) != in_L_other(s)]
print("strings where the two views disagree :", bad)
assert not bad
print("even #0 and even #1  <=>  even length and even #0.  Cross-check passed.")

View 3 agrees with view 2 &mdash; including the empty string, the usual trap.

In [ ]:
for s in positives: assert in_L_setcomp(s), s
for s in negatives: assert not in_L_setcomp(s), s
print("epsilon in L?", in_L_setcomp(''), " (0 zeros and 0 ones -- both even)")
print("all %d positive and %d negative examples agree with the comprehension"
      % (len(positives), len(negatives)))

And *now* the DFA, checked against all four.

In [ ]:
D = md2mc('''DFA
IF : 0 -> E0     !! even 1s, odd 0s
IF : 1 -> E1     !! odd 1s, even 0s
E0 : 0 -> IF
E0 : 1 -> E01
E1 : 0 -> E01
E1 : 1 -> IF
E01: 0 -> E1
E01: 1 -> E0
''')
assert all(accepts_dfa(D, s) == in_L_setcomp(s) for s in upto(10))
print("DFA agrees with the specification on all 2047 strings up to length 10")

## 4. Exercises


1. Write a fifth view of $L$ as a regular expression. Does it agree?
2. Find an English sentence for $L$ that is genuinely ambiguous. Which view catches it?
3. Which of the four views is easiest to *automate* as a check? Why?

In [ ]:
# Your work for the exercises above.